# Expert 05 — Testing & Code Quality


## 1. Writing Testable Code

In [ ]:
# Module under test
from typing import Optional

class Calculator:
    def add(self, a: float, b: float) -> float:
        return a + b

    def subtract(self, a: float, b: float) -> float:
        return a - b

    def multiply(self, a: float, b: float) -> float:
        return a * b

    def divide(self, a: float, b: float) -> float:
        if b == 0:
            raise ZeroDivisionError('Cannot divide by zero')
        return a / b

    def power(self, base: float, exp: int) -> float:
        if not isinstance(exp, int):
            raise TypeError(f'Exponent must be int, got {type(exp).__name__}')
        return base ** exp

calc = Calculator()
print(calc.add(2, 3))
print(calc.divide(10, 4))

## 2. pytest-style Tests (runnable inline)

In [ ]:
import traceback

def run_test(name, func):
    try:
        func()
        print(f'  PASS  {name}')
    except AssertionError as e:
        print(f'  FAIL  {name}: {e}')
    except Exception as e:
        print(f'  ERROR {name}: {e}')

calc = Calculator()

tests = [
    ('add positive',    lambda: assert_eq(calc.add(2, 3), 5)),
    ('add negative',    lambda: assert_eq(calc.add(-1, -1), -2)),
    ('add floats',      lambda: assert_approx(calc.add(0.1, 0.2), 0.3)),
    ('divide normal',   lambda: assert_eq(calc.divide(10, 2), 5.0)),
    ('divide by zero',  lambda: assert_raises(ZeroDivisionError, calc.divide, 10, 0)),
    ('power int',       lambda: assert_eq(calc.power(2, 10), 1024)),
    ('power bad type',  lambda: assert_raises(TypeError, calc.power, 2, 2.5)),
]

def assert_eq(a, b): assert a == b, f'{a!r} != {b!r}'
def assert_approx(a, b, tol=1e-9): assert abs(a-b) < tol, f'{a} not approx {b}'
def assert_raises(exc, func, *args):
    try:
        func(*args)
        raise AssertionError(f'Expected {exc.__name__} not raised')
    except exc:
        pass

print('Running tests:')
for name, test in tests:
    run_test(name, test)

## 3. Mocking

In [ ]:
from unittest.mock import Mock, MagicMock, patch, call

# Basic Mock
mock_db = Mock()
mock_db.get_user.return_value = {'id': 1, 'name': 'Alice'}

result = mock_db.get_user(1)
print('Result:', result)
mock_db.get_user.assert_called_once_with(1)
print('Call count:', mock_db.get_user.call_count)

# side_effect — different return per call
mock_db.get_user.side_effect = [
    {'id': 1, 'name': 'Alice'},
    {'id': 2, 'name': 'Bob'},
    Exception('DB connection lost'),
]

print(mock_db.get_user(1))
print(mock_db.get_user(2))
try:
    mock_db.get_user(3)
except Exception as e:
    print(f'Exception: {e}')

In [ ]:
# patch — replace in module namespace
import json
from unittest.mock import patch

def fetch_config(url):
    """Fetches config from URL."""
    import urllib.request
    with urllib.request.urlopen(url) as resp:
        return json.loads(resp.read())

# Test without hitting network
with patch('urllib.request.urlopen') as mock_urlopen:
    mock_response = MagicMock()
    mock_response.__enter__ = lambda s: s
    mock_response.__exit__ = Mock(return_value=False)
    mock_response.read.return_value = b'{"debug": true, "version": "1.0"}'
    mock_urlopen.return_value = mock_response

    config = fetch_config('http://example.com/config.json')
    print('Config:', config)
    mock_urlopen.assert_called_once_with('http://example.com/config.json')

## 4. Parametrized Testing Pattern

In [ ]:
# Simulate pytest.mark.parametrize
test_cases = [
    # (a, b, expected, description)
    (2,   3,   5,   'positive integers'),
    (-1, -1,  -2,   'negative integers'),
    (0,   0,   0,   'zeros'),
    (0.1, 0.2, 0.3, 'floats'),
    (100, -50, 50,  'mixed signs'),
]

calc = Calculator()
print('Parametrized add tests:')
for a, b, expected, desc in test_cases:
    result = calc.add(a, b)
    ok = abs(result - expected) < 1e-9
    status = 'PASS' if ok else 'FAIL'
    print(f'  {status}  add({a}, {b}) = {result} (expected {expected}) [{desc}]')

## 5. Test Fixtures Pattern

In [ ]:
import contextlib
import tempfile
from pathlib import Path

@contextlib.contextmanager
def temp_directory():
    """Fixture: temporary directory, cleaned up after test."""
    with tempfile.TemporaryDirectory() as tmpdir:
        yield Path(tmpdir)

@contextlib.contextmanager
def temp_file(content='', suffix='.txt'):
    """Fixture: temporary file with content."""
    with tempfile.NamedTemporaryFile(mode='w', suffix=suffix, delete=False) as f:
        f.write(content)
        path = Path(f.name)
    try:
        yield path
    finally:
        path.unlink(missing_ok=True)

# Use fixtures
with temp_directory() as tmpdir:
    (tmpdir / 'test.txt').write_text('hello')
    files = list(tmpdir.glob('*.txt'))
    print(f'Files in temp dir: {[f.name for f in files]}')

with temp_file('line1\nline2\nline3') as f:
    lines = f.read_text().splitlines()
    print(f'Lines: {lines}')

## 6. Code Quality: Type Hints + Validation

In [ ]:
from typing import TypeVar, Generic, Optional, Union
from dataclasses import dataclass, field

T = TypeVar('T')

@dataclass
class Result(Generic[T]):
    """Result type — either success or error."""
    value: Optional[T] = None
    error: Optional[str] = None

    @property
    def ok(self) -> bool:
        return self.error is None

    @classmethod
    def success(cls, value: T) -> 'Result[T]':
        return cls(value=value)

    @classmethod
    def failure(cls, error: str) -> 'Result[T]':
        return cls(error=error)

    def map(self, func):
        if self.ok:
            return Result.success(func(self.value))
        return self

def safe_divide(a: float, b: float) -> Result[float]:
    if b == 0:
        return Result.failure('Division by zero')
    return Result.success(a / b)

# Test
r1 = safe_divide(10, 2)
r2 = safe_divide(10, 0)

print(f'10/2: ok={r1.ok}, value={r1.value}')
print(f'10/0: ok={r2.ok}, error={r2.error}')

# Chain operations
result = safe_divide(100, 4).map(lambda x: x + 1).map(str)
print(f'Chained: {result.value}')